In [1]:
import sys, scanpy as sc
print(sys.version)
print(sc.__version__)

3.12.15 (main, Oct  9 2026, 15:21:55) [MSC v.1944 64 bit (AMD64)]
1.12.4


C:\Users\hp\AppData\Local\Temp\ipykernel_15064\3898917610.py:3: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(sc.__version__)


In [2]:
from pathlib import Path
import os

print(os.getcwd())
DATA = Path("../data")
f_tpm  = DATA / "GSE120575_Sade_Feldman_melanoma_single_cells_TPM_GEO.txt.gz"
f_meta = DATA / "GSE120575_patient_ID_single_cells.txt.gz"
print(f_tpm.exists(), f_meta.exists())

f:\melanoma\melanoma-scrna-analysis\notebooks
True True


In [4]:
import gzip
with gzip.open(f_meta, "rt") as fh:
    for i, line in enumerate(fh):
        print(i, line[:150].rstrip())
        if i >= 30: break

0 # High-throughput sequencing metadata template (version 2.1).
1 # All fields in this template must be completed.
2 # Templates containing example data are found in the METADATA EXAMPLES spreadsheet tabs at the foot of this page.
3 # Field names (in blue on this page) should not be edited. Hover over cells containing field names to view field content guidelines.
4 "# Human data. If there are patient privacy concerns regarding making data fully public through GEO, please submit to NCBI's dbGaP (http://www.ncbi.nl
5 
6 SERIES
7 # This section describes the overall experiment.
8 title	Defining T cell states associated with response to checkpoint immunotherapy in melanoma
9 summary	"we profiled 16,291 immune cells from 48 tumor samples of melanoma patients treated with checkpoint inhibitors, using single-cell transcriptom
10 overall design	"Tumors were disaggregated, sorted into CD45+ single cells, and profiled by Smart-seq2"
11 contributor	Moshe Sade-Feldman
12 contributor	Keren Yizhak
1

In [5]:
with gzip.open(f_tpm, "rt") as fh:
    for i, line in enumerate(fh):
        print(i, len(line.split("\t")), line[:300].rstrip())
        if i >= 2: break

0 16292 	A10_P3_M11	A11_P1_M11	A11_P3_M11	A11_P4_M11	A12_P3_M11	A12_P6_M11	A2_P1_M11	A2_P4_M11	A3_P1_M11	A3_P3_M11	A4_P3_M11	A4_P4_M11	A4_P6_M11	A5_P4_M11	A5_P5_M11	A6_P1_M11	A6_P4_M11	A6_P6_M11	A7_P2_M11	A7_P6_M11	A8_P1_M11	A8_P6_M11	A9_P1_M11	A9_P4_M11	B10_P1_M11	B10_P3_M11	B10_P4_M11	B10_P6_M11	B11_P6_M1
1 16292 	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P
2 16293 TSPAN6	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.0


In [6]:
with gzip.open(f_tpm, "rt") as fh:
    for i, line in enumerate(fh):
        parts = line.rstrip("\n").split("\t")
        print(i, "تعداد فیلد:", len(parts))
        print("   اولین فیلدها:", [repr(p) for p in parts[:3]])
        print("   آخرین فیلدها:", [repr(p) for p in parts[-3:]])
        if i >= 4:
            break

0 تعداد فیلد: 16292
   اولین فیلدها: ["''", "'A10_P3_M11'", "'A11_P1_M11'"]
   آخرین فیلدها: ["'H7_P5_M67_L001_T_enriched'", "'H8_P5_M67_L001_T_enriched'", "'H9_P5_M67_L001_T_enriched'"]
1 تعداد فیلد: 16292
   اولین فیلدها: ["''", "'Pre_P1'", "'Pre_P1'"]
   آخرین فیلدها: ["'Post_P6_T_enriched'", "'Post_P6_T_enriched'", "'Post_P6_T_enriched'"]
2 تعداد فیلد: 16293
   اولین فیلدها: ["'TSPAN6'", "'0.00'", "'0.00'"]
   آخرین فیلدها: ["'0.00'", "'0.00'", "''"]
3 تعداد فیلد: 16293
   اولین فیلدها: ["'TNMD'", "'0.00'", "'0.00'"]
   آخرین فیلدها: ["'0.00'", "'0.00'", "''"]
4 تعداد فیلد: 16293
   اولین فیلدها: ["'DPM1'", "'0.00'", "'9.24'"]
   آخرین فیلدها: ["'0.00'", "'0.00'", "''"]


In [ ]:
import pandas as pd

meta = pd.read_csv(f_meta, sep="\t", skiprows=19, nrows=16291, encoding="latin-1")
meta = meta.loc[:, ~meta.columns.str.startswith("Unnamed")]
cols = meta.columns.tolist()
print(cols)

meta = meta.rename(columns={
    cols[1]: "cell_id",
    cols[4]: "sample",
    cols[5]: "response",
    cols[6]: "therapy",
})

for c in ["cell_id", "sample", "response", "therapy"]:
    meta[c] = meta[c].str.strip()

meta = meta.set_index("cell_id")
meta["timepoint"]  = meta["sample"].str.split("_").str[0]
meta["patient"]    = meta["sample"].str.split("_").str[1]
meta["T_enriched"] = meta.index.str.contains("T_enriched")

print(meta.shape)
print(meta.index.is_unique)
print(meta["sample"].nunique(), meta["patient"].nunique())
print(meta["response"].value_counts())
print(meta["therapy"].value_counts())
print(meta.groupby("patient")["response"].nunique().max())
print(meta["T_enriched"].value_counts())
print(meta[["sample", "response", "therapy"]].tail(3))


['Sample name', 'title', 'source name', 'organism', 'characteristics: patinet ID (Pre=baseline; Post= on treatment)', 'characteristics: response', 'characteristics: therapy', 'molecule', 'description', 'processed data file ', 'raw file']
(16291, 13)
True
48 32
response
Non-responder    10727
Responder         5564
Name: count, dtype: int64
therapy
anti-PD1          11653
anti-CTLA4+PD1     4121
anti-CTLA4          517
Name: count, dtype: int64
2
T_enriched
False    15418
True       873
Name: count, dtype: int64
                            sample       response   therapy
cell_id                                                    
H7_P5_M67_L001_T_enriched  Post_P6  Non-responder  anti-PD1
H8_P5_M67_L001_T_enriched  Post_P6  Non-responder  anti-PD1
H9_P5_M67_L001_T_enriched  Post_P6  Non-responder  anti-PD1


In [15]:
meta = meta.loc[:, ~meta.columns.str.startswith("Unnamed")]

meta["T_enriched"] = meta.index.str.contains("T_enriched")
print(meta["T_enriched"].value_counts())

ct = pd.crosstab(meta["sample"], meta["T_enriched"])
print(ct[ct.get(True, 0) > 0])

n_lab = meta.groupby("patient")["response"].nunique()
mixed = n_lab[n_lab > 1].index.tolist()
print("بیماران با دو برچسب:", mixed)
print(meta[meta["patient"].isin(mixed)]
      .groupby(["patient", "sample", "response", "therapy"]).size())

T_enriched
False    15418
True       873
Name: count, dtype: int64
T_enriched  False  True 
sample                  
Post_P10      308     88
Post_P13      265    184
Post_P16      369     94
Post_P2       301     79
Post_P4       421    254
Post_P6       308     92
Post_P8       327     82
بیماران با دو برچسب: ['P1', 'P28', 'P4', 'P5']
patient  sample      response       therapy       
P1       Post_P1     Responder      anti-PD1          291
         Post_P1_2   Non-responder  anti-PD1          353
         Pre_P1      Responder      anti-CTLA4        229
P28      Post_P28    Non-responder  anti-CTLA4+PD1    369
         Post_P28_2  Non-responder  anti-CTLA4+PD1    372
         Pre_P28     Responder      anti-CTLA4+PD1    337
P4       Post_P4     Responder      anti-CTLA4+PD1    675
         Pre_P4      Non-responder  anti-CTLA4+PD1    311
P5       Post_P5     Non-responder  anti-PD1          292
         Post_P5_2   Responder      anti-PD1          336
dtype: int64


In [12]:
extra = ["molecule", "description", "processed data file ", "raw file"]
print(meta[extra].nunique())
print(meta[extra].drop_duplicates().head())

molecule                0
description             0
processed data file     0
raw file                0
dtype: int64
            molecule  description  processed data file   raw file
cell_id                                                          
A10_P3_M11       NaN          NaN                   NaN       NaN


In [13]:
print(meta[["Sample name", "source name", "organism"]].nunique())

keep = ["sample", "response", "therapy", "timepoint", "patient", "T_enriched"]
meta = meta[keep].copy()

print(meta.shape)
print(meta.head(3))

Sample name    16291
source name        1
organism           1
dtype: int64
(16291, 6)
            sample   response     therapy timepoint patient  T_enriched
cell_id                                                                
A10_P3_M11  Pre_P1  Responder  anti-CTLA4       Pre      P1       False
A11_P1_M11  Pre_P1  Responder  anti-CTLA4       Pre      P1       False
A11_P3_M11  Pre_P1  Responder  anti-CTLA4       Pre      P1       False


In [16]:
import gzip
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad

# 1) Read the two header lines (cell IDs and per-cell sample labels)
with gzip.open(f_tpm, "rt") as fh:
    cell_ids      = fh.readline().rstrip("\n").split("\t")[1:]   # [1:] skips the empty first field
    sample_labels = fh.readline().rstrip("\n").split("\t")[1:]
n_cells = len(cell_ids)
print(n_cells, len(sample_labels))

# 2) Read the gene rows in chunks and store each chunk as a sparse matrix
blocks, genes = [], []
reader = pd.read_csv(f_tpm, sep="\t", skiprows=2, header=None, index_col=0,
                     usecols=range(n_cells + 1),   # gene name + cell columns; drops the trailing empty field
                     chunksize=2000)               # 2000 genes at a time to avoid running out of RAM
for ch in reader:
    genes.extend(ch.index.tolist())
    blocks.append(sp.csr_matrix(ch.to_numpy(dtype="float32")))
    if len(blocks) % 5 == 0:
        print(len(genes), "genes read")

# 3) Build the AnnData object (rows = cells, columns = genes)
X = sp.vstack(blocks).T.tocsr()   # stack chunks (genes x cells), then transpose to cells x genes
adata = ad.AnnData(
    X,
    obs=pd.DataFrame({"matrix_sample": sample_labels}, index=cell_ids),   # keep the matrix sample label for a later consistency check
    var=pd.DataFrame(index=genes),
)
adata.var_names_make_unique()   # duplicated gene names would break scanpy later

# 4) Sanity checks
print(adata.shape, adata.obs_names.is_unique)
print("max:", adata.X.max(), "| min:", adata.X.min())
print("percent zeros:", round(100 * (1 - adata.X.nnz / (adata.n_obs * adata.n_vars)), 1))
print(pd.Series(np.asarray(adata.X.sum(axis=1)).ravel()).describe())   # per-cell total TPM, expected near 1e6

16291 16291
10000 genes read
20000 genes read
30000 genes read
40000 genes read
50000 genes read
(16291, 55737) True
max: 19.42 | min: 0.0
percent zeros: 95.7
count    16291.000000
mean     15328.044922
std       4740.536621
min       5147.560059
25%      12087.844727
50%      14193.959961
75%      17230.519531
max      44286.832031
dtype: float64


In [20]:
# Checkpoint: save the raw matrix so it never has to be re-read from the .gz file
adata.write_h5ad(DATA / "GSE120575_raw_matrix.h5ad")

In [19]:
# Candidate transforms: log_base(x + 1)
LOG_CANDIDATES = {"log2(x+1)": 2.0, "ln(x+1)": np.e, "log10(x+1)": 10.0}

def to_linear(X, base):
    """Invert log_base(x + 1) on a sparse matrix (returns a float64 copy)."""
    L = X.astype("float64")
    L.data = np.expm1(L.data * np.log(base))   # = base**x - 1
    return L

# Diagnostic on a random subset of cells: the right base gives near-constant linear totals
rng = np.random.default_rng(0)
idx = rng.choice(adata.n_obs, size=1000, replace=False)
X_sub = adata.X[idx]
for name, base in LOG_CANDIDATES.items():
    s = np.asarray(to_linear(X_sub, base).sum(axis=1)).ravel()
    print(f"{name:12s} mean total = {s.mean():16.1f} | CV = {s.std() / s.mean():.4f}")

# Result: log2(x+1) -> mean total ~ 1e6 and CV 0.0024, so the matrix is log2(TPM + 1)
LOG_BASE = 2

log2(x+1)    mean total =         994509.7 | CV = 0.0024
ln(x+1)      mean total =       45531761.6 | CV = 0.5054
log10(x+1)   mean total = 157822486010497472.0 | CV = 4.6658


In [21]:
# Attach per-cell metadata (joined on the cell ID)
adata.obs = adata.obs.join(
    meta[["sample", "response", "therapy", "timepoint", "patient", "T_enriched"]]
)
print(adata.obs.isna().sum())

# The matrix header labels T-enriched samples like "Post_P6_T_enriched"; metadata has no suffix
lab = adata.obs["matrix_sample"].str.replace("_T_enriched", "", regex=False)
adata.obs["matrix_timepoint"] = lab.str.split("_").str[0]
adata.obs["matrix_patient"]   = lab.str.split("_").str[1]

ok_tp     = (adata.obs["matrix_timepoint"] == adata.obs["timepoint"]).mean()
ok_pat    = (adata.obs["matrix_patient"]   == adata.obs["patient"]).mean()
ok_sample = (lab == adata.obs["sample"]).mean()
print("timepoint match:", ok_tp, "| patient match:", ok_pat, "| exact sample match:", ok_sample)
assert ok_tp == 1.0 and ok_pat == 1.0, "cell order differs between matrix and metadata"

# Show any sample-label differences (e.g. a possible _2 suffix)
print(adata.obs.loc[lab != adata.obs["sample"], ["matrix_sample", "sample"]].drop_duplicates())

matrix_sample    0
sample           0
response         0
therapy          0
timepoint        0
patient          0
T_enriched       0
dtype: int64
timepoint match: 1.0 | patient match: 1.0 | exact sample match: 0.9927567368485667
                                               matrix_sample    sample
A3_P7_MMD9_L001_myeloid_enriched   Post_P17_myeloid_enriched  Post_P17
A10_P7_MMD7_L001_myeloid_enriched  Post_P19_myeloid_enriched  Post_P19


In [22]:
# The sorting strategy is encoded in the cell ID, e.g. "..._L001_T_enriched" or "..._L001_myeloid_enriched"
enr = adata.obs_names.str.extract(r"_([A-Za-z]+)_enriched$")[0]
adata.obs["enrichment"]  = enr.fillna("none").to_numpy()   # "T", "myeloid" or "none"
adata.obs["is_enriched"] = adata.obs["enrichment"] != "none"
print(adata.obs["enrichment"].value_counts())

# Which samples contain enriched cells?
ct = pd.crosstab(adata.obs["sample"], adata.obs["enrichment"])
print(ct[ct.drop(columns="none").sum(axis=1) > 0])

# Sanity checks
print("agrees with the earlier T_enriched flag:",
      ((adata.obs["enrichment"] == "T") == adata.obs["T_enriched"]).all())
n_word = int(adata.obs_names.str.contains("enriched").sum())
print("IDs containing 'enriched':", n_word, "| flagged:", int(adata.obs["is_enriched"].sum()))
assert n_word == int(adata.obs["is_enriched"].sum()), "an 'enriched' label was missed by the pattern"

enrichment
none       15300
T            873
myeloid      118
Name: count, dtype: int64
enrichment    T  myeloid  none
sample                        
Post_P10     88        0   308
Post_P13    184        0   265
Post_P16     94        0   369
Post_P17      0       43   238
Post_P19      0       75   328
Post_P2      79        0   301
Post_P4     254        0   421
Post_P6      92        0   308
Post_P8      82        0   327
agrees with the earlier T_enriched flag: True
IDs containing 'enriched': 991 | flagged: 991


In [23]:
EXCLUDE_PATIENTS = ["P1", "P28"]   # acquired resistance (responded first, then relapsed)

n_before = adata.n_obs
keep_mask = ~adata.obs["patient"].isin(EXCLUDE_PATIENTS)
excluded_ids = adata.obs_names[~keep_mask].tolist()   # kept for a later sensitivity analysis

print(adata.obs.loc[~keep_mask, "patient"].value_counts())
adata = adata[keep_mask].copy()

print("cells before:", n_before, "| after:", adata.n_obs, "| removed:", len(excluded_ids))
print("patients:", adata.obs["patient"].nunique(), "| samples:", adata.obs["sample"].nunique())
print(adata.obs.drop_duplicates("sample")["timepoint"].value_counts())
print("enriched cells kept:", int(adata.obs["is_enriched"].sum()))

patient
P28    1078
P1      873
Name: count, dtype: int64
cells before: 16291 | after: 14340 | removed: 1951
patients: 30 | samples: 42
timepoint
Post    25
Pre     17
Name: count, dtype: int64
enriched cells kept: 991


In [24]:
# Patient-level clinical table (built from GEO + the paper's supplement)
meta_pat = pd.read_excel(DATA / "metadata.xlsx")
meta_pat = meta_pat.rename(columns={"patient_id": "patient"}).set_index("patient")

# One row per sample from the per-cell metadata
sample_tbl = (meta.reset_index().groupby("sample")
              .agg(patient=("patient", "first"), timepoint=("timepoint", "first"),
                   response=("response", "first"), therapy=("therapy", "first"),
                   n_resp=("response", "nunique"), n_cells=("cell_id", "size"))
              .reset_index())
print("samples with more than one response label:", (sample_tbl.n_resp > 1).sum())

# Reference sample per patient: the Pre sample if it exists, otherwise the first Post sample
sample_tbl["order"] = (sample_tbl["timepoint"] != "Pre").astype(int)
ref = sample_tbl.sort_values(["patient", "order", "sample"]).groupby("patient").first()
n_samp = sample_tbl.groupby("patient").size().rename("n_samples_cells")

cmp = (ref[["response", "therapy"]]
       .join(meta_pat[["response_geo", "therapy", "n_samples"]], rsuffix="_xlsx")
       .join(n_samp))

print("patients in only one source:", set(ref.index) ^ set(meta_pat.index))
print("response == response_geo:", (cmp["response"] == cmp["response_geo"]).all())
print("therapy == xlsx therapy:", (cmp["therapy"] == cmp["therapy_xlsx"]).all())
print("sample counts match:", (cmp["n_samples_cells"] == cmp["n_samples"]).all())
print(cmp[(cmp["response"] != cmp["response_geo"]) | (cmp["therapy"] != cmp["therapy_xlsx"])])

samples with more than one response label: 0
patients in only one source: set()
response == response_geo: True
therapy == xlsx therapy: True
sample counts match: True
Empty DataFrame
Columns: [response, therapy, response_geo, therapy_xlsx, n_samples, n_samples_cells]
Index: []


In [25]:
cols = ["age", "gender", "response_final", "response_3group",
        "OS_days", "status", "therapy"]
clin = meta_pat[cols].rename(columns={"therapy": "therapy_patient"})
adata.obs = adata.obs.join(clin, on="patient")

# Sample-level labels are kept under explicit names; "response" now means the patient-level label
adata.obs = adata.obs.rename(columns={"response": "response_sample",
                                      "therapy":  "therapy_sample"})
adata.obs["response"] = adata.obs["response_final"]

print(adata.obs[["response_final", "response_3group", "age"]].isna().sum())
print(adata.obs.drop_duplicates("patient")["response"].value_counts())
assert set(adata.obs["response"].unique()) == {"Responder", "Non-responder"}

adata.write_h5ad(DATA / "GSE120575_clinical.h5ad")

response_final     0
response_3group    0
age                0
dtype: int64
response
Non-responder    19
Responder        11
Name: count, dtype: int64
